# Notebook chạy lại static baseline và dynamic prototype

Notebook này gom các bước chạy lại repo theo đúng flow hiện tại.

Bạn có thể dùng notebook này theo 2 hướng:
- static baseline: dùng dataset tĩnh để demo nhanh, ổn định;
- dynamic prototype: sinh claim động theo taxonomy của FACT-AUDIT khi có model thật như Gemini.

## Bước 1: xác định thư mục làm việc

Phần này làm gì:
- Đảm bảo notebook luôn chạy trong đúng thư mục `fact_audit_reproduction/`.
- Tránh lỗi đường dẫn khi bạn mở notebook từ thư mục khác.

Output kỳ vọng:
- In ra dòng `Working directory: .../fact_audit_reproduction`.

In [ ]:
from pathlib import Path
import os

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
elif root.name != "fact_audit_reproduction":
    candidate = root / "fact_audit_reproduction"
    if candidate.exists():
        root = candidate

os.chdir(root)
print("Working directory:", Path.cwd())

## Bước 2: hàm chạy lệnh

Phần này làm gì:
- Tạo hàm `run_command(...)` để chạy script bằng Python.
- In lại stdout/stderr ngay trong notebook để dễ theo dõi.

Output kỳ vọng:
- Không có file output mới.
- Chỉ định nghĩa helper function để dùng cho các bước sau.

In [ ]:
import subprocess
import sys

def run_command(args):
    print("$", " ".join(args))
    completed = subprocess.run(args, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    completed.check_returncode()
    return completed

## Bước 3: tạo claim set 30 mẫu

Phần này làm gì:
- Chạy `scripts/make_claim_set.py`.
- Lấy dữ liệu từ `data/source/fact_checking_normalized.jsonl`.
- Tạo một claim set tĩnh gồm 30 mẫu để baseline và RAG dùng chung.

Output kỳ vọng:
- Tạo hoặc ghi đè `data/claim_sets/claim_set_30.jsonl`.
- Terminal in ra kiểu `Wrote 30 claims ...`.

In [ ]:
run_command([sys.executable, "scripts/make_claim_set.py", "--size", "30"])

## Bước 4: chạy smoke test 3 claims

Phần này làm gì:
- Chạy `scripts/run_smoke_test.py`.
- Dùng 3 claim đầu để kiểm tra pipeline end-to-end có chạy được không.

Output kỳ vọng:
- `outputs/smoke_test.jsonl`
- `outputs/smoke_test_scores.csv`
- Terminal in ra số dòng và average score.

In [ ]:
run_command([sys.executable, "scripts/run_smoke_test.py"])

## Bước 5: chạy baseline đầy đủ 30 claims

Phần này làm gì:
- Chạy `scripts/run_baseline.py` trên toàn bộ `claim_set_30.jsonl`.
- Mặc định dùng provider trong `config.yaml`.
- Nếu chưa đổi config thì provider hiện tại thường là `mock`.

Output kỳ vọng:
- `outputs/baseline_results.jsonl`
- `outputs/scores.csv`
- Terminal in ra số verdict đúng và average score.

In [ ]:
run_command([sys.executable, "scripts/run_baseline.py"])

## Bước 6: chạy demo cache 5 claims

Phần này làm gì:
- Chạy `scripts/run_baseline_demo.py --use-cache --limit 5`.
- Phù hợp khi demo vì có thể dùng lại output có sẵn thay vì gọi model lại.

Output kỳ vọng:
- `outputs/cached_demo/baseline_demo_results.jsonl`
- `outputs/cached_demo/demo_scores.csv`
- Nếu cache đã có, terminal sẽ báo `Using cached outputs ...`.

In [ ]:
run_command([sys.executable, "scripts/run_baseline_demo.py", "--use-cache", "--limit", "5"])

## Bước 7: xem nhanh output JSONL/CSV

Phần này làm gì:
- Mở nhanh `outputs/baseline_results.jsonl`.
- In 2 dòng mẫu để kiểm tra `claim_id`, `verdict`, `score`, `provider`.
- In header của `outputs/scores.csv` để kiểm tra đủ cột yêu cầu.

Output kỳ vọng:
- In 2 sample rows từ baseline JSONL.
- In danh sách cột của file CSV.

In [ ]:
import csv
import json
from itertools import islice

baseline_path = Path("outputs/baseline_results.jsonl")
scores_path = Path("outputs/scores.csv")

print("Baseline sample:")
with baseline_path.open(encoding="utf-8") as handle:
    for line in islice(handle, 2):
        row = json.loads(line)
        print({
            "claim_id": row["claim_id"],
            "verdict": row["verdict"],
            "score": row["score"],
            "provider": row["provider"],
        })

print("\nCSV header:")
with scores_path.open(encoding="utf-8") as handle:
    reader = csv.reader(handle)
    print(next(reader))

## Bước 8: chạy static baseline với Gemini

Phần này làm gì:
- Chạy baseline tĩnh nhưng đổi provider sang `gemini`.
- Dùng khi bạn muốn có output tốt hơn `mock`.

Điều kiện trước khi chạy:
- Trong `.env` phải có `GEMINI_API_KEY`.
- Model mặc định nên là `gemini-2.5-flash`.

Output kỳ vọng:
- Ghi đè `outputs/baseline_results.jsonl` và `outputs/scores.csv` bằng kết quả từ Gemini.

In [ ]:
# run_command([
#     sys.executable,
#     "scripts/run_baseline.py",
#     "--provider",
#     "gemini",
#     "--model",
#     "gemini-2.5-flash",
# ])

## Bước 9: sinh claim set động theo taxonomy

Phần này làm gì:
- Chạy `scripts/generate_dynamic_claim_set.py`.
- Đọc taxonomy từ `external/FACT-AUDIT/data/fact_cat.json`.
- Nhờ model sinh claim mới theo từng scenario thay vì lấy trực tiếp từ dataset tĩnh.

Điều kiện trước khi chạy:
- Không dùng `mock` cho bước này.
- Nên dùng `gemini`, `openai`, hoặc `transformers`.

Output kỳ vọng:
- Tạo `data/claim_sets/dynamic_claim_set_12.jsonl`.

In [ ]:
# run_command([
#     sys.executable,
#     "scripts/generate_dynamic_claim_set.py",
#     "--provider",
#     "gemini",
#     "--model",
#     "gemini-2.5-flash",
#     "--size",
#     "12",
# ])

## Bước 10: chạy baseline trên claim động

Phần này làm gì:
- Lấy file `data/claim_sets/dynamic_claim_set_12.jsonl` vừa sinh ở bước trước.
- Chạy `scripts/run_baseline.py` trên claim set động đó.
- Bước này giúp bạn thấy output gần tinh thần FACT-AUDIT hơn so với static baseline.

Output kỳ vọng:
- Ghi đè `outputs/baseline_results.jsonl` và `outputs/scores.csv` bằng kết quả trên claim động.
- Terminal in ra số dòng, số verdict đúng và average score.

In [ ]:
# run_command([
#     sys.executable,
#     "scripts/run_baseline.py",
#     "--input-file",
#     "data/claim_sets/dynamic_claim_set_12.jsonl",
#     "--provider",
#     "gemini",
#     "--model",
#     "gemini-2.5-flash",
# ])